# ⚖️ THEMIS v1 — Proof of Pipeline

**The Parametric Legal Intelligence Engine for Indian Law**

![Model](https://img.shields.io/badge/model-Mistral%207B%20LoRA-blueviolet)
![Status](https://img.shields.io/badge/status-v1%20trained%20%7C%20v2%20in%20progress-orange)
![License](https://img.shields.io/badge/license-MIT-lightgrey)

---

## What this notebook demonstrates

This notebook runs **THEMIS v1** — a LoRA adapter fine-tuned on 1,939 Indian legal Q&A pairs. It proves the **end-to-end pipeline works**: data scraping → synthetic generation → LoRA fine-tuning → HuggingFace deployment → inference.

### What v1 DOES demonstrate:
- ✅ End-to-end fine-tuning pipeline on Kaggle free T4 GPU
- ✅ LoRA adapter trained and published to HuggingFace Hub
- ✅ Alpaca instruction format learned — model responds in legal assistant style
- ✅ Disclaimer behavior trained correctly
- ✅ Response structure (citations, recommendations) partially learned

### What v1 does NOT demonstrate:
- ❌ Accurate section number citation — **hallucinates on specific queries**
- ❌ BNS abbreviation recognition — model confuses "BNS" with unrelated expansions
- ❌ Deep statutory knowledge — 1,939 pairs was insufficient for domain grounding
- ❌ IPC → BNS mapping — transition knowledge not retained at this scale

> **Root cause:** Mistral 7B has near-zero BNS 2023 pretraining knowledge — BNS was enacted Dec 2023, at/after Mistral’s training cutoff. The fine-tune taught the model *how to respond like a lawyer* but not *what Indian law says*.

### For best results in this notebook:
- Use **full act names** ("Bharatiya Nyaya Sanhita" not "BNS")
- Ask **general legal questions** not specific section numbers
- Treat outputs as **orientation**, never as authoritative legal reference

---

## v2 Roadmap

| Parameter | v1 (this) | v2 (next) |
|-----------|-----------|----------|
| Training pairs | 1,939 | 10,000–15,000 |
| LoRA rank | 8 | 16 |
| Target modules | q_proj, v_proj | q,k,v,o proj |
| Sequence length | 512 | 1,024 |
| Expected citation accuracy | ~40% | >70% |

---

## Setup (Kaggle)

1. ⚙️ Settings → Accelerator → **GPU T4 × 1**
2. ⚙️ Settings → Internet → **On**
3. Runtime → Run all cells

---

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers>=4.35.0 peft>=0.6.0 bitsandbytes>=0.41.0 accelerate>=0.21.0

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU. Enable GPU in Settings \u2192 Accelerator.")

---
## Cell 2 — Configuration

In [ ]:
BASE_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
LORA_ADAPTER = "Daniel2503/themis-mistral-7b-lora"

MAX_NEW_TOKENS = 512
TEMPERATURE = 0.3
TOP_P = 0.9
REPETITION_PENALTY = 1.1

INSTRUCTION_TEMPLATE = """### Instruction:
{question}

### Response:
"""

DISCLAIMER = (
    "\n\n[DISCLAIMER: This is legal orientation, not legal advice. "
    "Consult a qualified advocate for your specific situation. "
    "v1 hallucination rate on section-specific queries: ~60%.]"
)

print(f"Base model:   {BASE_MODEL}")
print(f"LoRA adapter: {LORA_ADAPTER}")

---
## Cell 3 — Load Model + LoRA Adapter

Downloads ~4GB base model + ~33MB adapter on first run. Cached afterward.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model: {BASE_MODEL}")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
print(f"Base model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

print(f"Attaching LoRA adapter: {LORA_ADAPTER}")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
model.eval()
print(f"\n✅ Ready. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

---
## Cell 4 — Inference Function

In [ ]:
def ask(question: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Ask a legal question. Use FULL ACT NAMES for best results."""
    prompt = INSTRUCTION_TEMPLATE.format(question=question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_tokens = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_tokens:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    if "disclaimer" not in response.lower():
        response += DISCLAIMER

    return response

print("ask() ready.")

---
## Cell 5 — Test with Good Questions (Full Act Names)

These questions use full act names, which v1 handles better than abbreviations.

In [ ]:
# @title Good questions (full act names)
good_questions = [
    "What is the punishment for theft under the Bharatiya Nyaya Sanhita?",
    "How do I file a complaint under the Consumer Protection Act?",
    "What are my rights under the Right to Information Act?",
]

for q in good_questions:
    print(f"\n❓ {q}")
    print("-" * 60)
    response = ask(q)
    print(response)
    print()

---
## Cell 6 — Test Known Failure Modes

These cells demonstrate v1’s known weaknesses. This is intentional — documenting failures is part of honest evaluation.

In [ ]:
# @title Failure Mode 1: BNS abbreviation confusion
print("Testing: Does the model know what 'BNS' means?\n")
response = ask("What is BNS?")
print(response)
print("\n⚠️  If the model says anything OTHER than 'Bharatiya Nyaya Sanhita', this is a known v1 failure.")

In [ ]:
# @title Failure Mode 2: Section number hallucination
print("Testing: Does the model hallucinate section numbers?\n")
response = ask("What does Section 303 of the Bharatiya Nyaya Sanhita say?")
print(response)
print("\n⚠️  Section 303 BNS is about theft. Verify the model’s answer against India Code.")

In [ ]:
# @title Failure Mode 3: IPC to BNS mapping
print("Testing: Can the model map IPC sections to BNS equivalents?\n")
response = ask("What is the BNS equivalent of Section 302 of the Indian Penal Code?")
print(response)
print("\n⚠️  IPC 302 (murder) maps to BNS 101. Check if the model gets this right.")

---
## Cell 7 — Custom Question

Try your own question. **Tip:** Use full act names for better results.

In [ ]:
# @title Ask your own question
question = "What is the punishment for murder under the Bharatiya Nyaya Sanhita?"  # @param {type:"string"}

print(f"❓ {question}\n")
response = ask(question)
print(response)

---
## Cell 8 — What This Demonstrates (Summary)

In [ ]:
print("=" * 60)
print("  THEMIS v1 — WHAT THIS NOTEBOOK DEMONSTRATES")
print("=" * 60)
print(""
print("PASS: End-to-end fine-tuning pipeline")
print("  Data scraping (India Code) → Synthetic generation")
print("  → LoRA training (Unsloth/Kaggle T4)")
print("  → HuggingFace Hub deployment")
print("  → PEFT inference with 4-bit quantization")
print("")
print("PASS: Instruction-following behavior")
print("  Model responds in legal assistant style")
print("  Disclaimer behavior trained correctly")
print("  Response structure (citations, recommendations) learned")
print("")
print("FAIL: Actual legal knowledge")
print("  ~60% hallucination rate on BNS-specific queries")
print("  BNS abbreviation not recognized")
print("  Section numbers frequently fabricated")
print("  IPC → BNS mapping not retained")
print("")
print("ROOT CAUSE: Mistral 7B has zero BNS 2023 pretraining data.")
print("The LoRA taught style, not substance. v2 targets 10-15k pairs")
print("with full statutory text to fix this.")
print("=" * 60)

---

## Performance (T4 GPU)

| Metric | Value |
|--------|-------|
| Model download | ~2 min |
| Model load | ~30 sec |
| Response (512 tokens) | ~5–8 sec |
| VRAM usage | ~4.5 GB |

## Project Links

| Resource | Link |
|----------|------|
| LoRA Adapter | [Daniel2503/themis-mistral-7b-lora](https://huggingface.co/Daniel2503/themis-mistral-7b-lora) |
| GitHub | [github.com/Daniel2503/themis](https://github.com/Daniel2503/themis) |
| v2 Roadmap | See README.md on GitHub |

---

*THEMIS v1 proves the pipeline works. v2 will make the model actually know Indian law.*